# DDPM on CelebA 64×64 


## 1. Setup

In [10]:
import os, math, copy, time, random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Device: cpu


## 2. Config

In [ ]:
IMG_SIZE   = 64
OUT_DIR    = '/kaggle/working/outputs'

BASE_CH       = 96            # U-Net width. 96 is a good speed/quality tradeoff 
CH_MULTS      = (1, 2, 2, 4)  # channels at each resolution: 96,192,192,384
ATTN_AT       = (16, 8)       # add self-attention at these spatial resolutions

T_STEPS    = 1000
BETA_START = 1e-4
BETA_END   = 0.02             # linear schedule — simple, matches original DDPM paper

BATCH_SIZE   = 32
LR           = 2e-4
NUM_EPOCHS   = 35
EMA_DECAY    = 0.999          # slightly lower than 0.9999 so EMA catches up faster early on
SAVE_EVERY   = 2              # checkpoint every 2 epochs
SAMPLE_EVERY = 2

# If resuming from a saved checkpoint
RESUME_FROM_DIR = None

os.makedirs(f'{OUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUT_DIR}/samples', exist_ok=True)

torch.manual_seed(0)
random.seed(0)

## 3. Dataset

In [ ]:
transform = T.Compose([
    T.CenterCrop(140),
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),   # -> [-1, 1]
])

class CelebAFolder(Dataset):
    """Reads jpgs directly from a folder. Adjust ROOT below to match your
    Kaggle dataset's actual path."""
    def __init__(self, root, transform):
        self.files = sorted(Path(root).glob('*.jpg'))
        if len(self.files) == 0:
            raise FileNotFoundError(
                f'No .jpg files found in {root}. '
                )
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        return self.transform(img)


CELEBA_ROOT = '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'

if CELEBA_ROOT is None:
    raise FileNotFoundError(
        'Could not auto-find CelebA images under /kaggle/input. '
        'Add the celeba-dataset via Add Data, then re-run this cell.')

print('Found CelebA images at:', CELEBA_ROOT)
train_ds = CelebAFolder(CELEBA_ROOT, transform)
print(f'Dataset size: {len(train_ds):,} images')

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)
print(f'Batches per epoch: {len(train_dl)}')

Found CelebA images at: /kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba


FileNotFoundError: No .jpg files found in /kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba. Run the os.walk snippet below to find the correct path.

## 4. Diffusion Schedule (linear — simple, matches original DDPM paper)

In [ ]:
betas = torch.linspace(BETA_START, BETA_END, T_STEPS, device=DEVICE)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=DEVICE), alphas_cumprod[:-1]])

sqrt_alphas_cumprod = alphas_cumprod.sqrt()
sqrt_one_minus_alphas_cumprod = (1 - alphas_cumprod).sqrt()
sqrt_recip_alphas = (1.0 / alphas).sqrt()
posterior_variance = betas * (1 - alphas_cumprod_prev) / (1 - alphas_cumprod)

def extract(coeffs, t, shape):
    """Pull out per-sample coefficients for timestep t and reshape to broadcast
    against an image tensor [B,C,H,W]."""
    out = coeffs.gather(0, t)
    return out.reshape(-1, 1, 1, 1)

def q_sample(x0, t, noise):
    """Forward process: produce x_t directly from x_0 in one step."""
    a = extract(sqrt_alphas_cumprod, t, x0.shape)
    b = extract(sqrt_one_minus_alphas_cumprod, t, x0.shape)
    return a * x0 + b * noise

@torch.no_grad()
def p_sample_step(model, x, t_int):
    """One reverse step: x_t -> x_{t-1}. model predicts noise (epsilon)."""
    B = x.shape[0]
    t = torch.full((B,), t_int, device=DEVICE, dtype=torch.long)
    eps_pred = model(x, t)

    beta_t = extract(betas, t, x.shape)
    sqrt_one_minus_acp_t = extract(sqrt_one_minus_alphas_cumprod, t, x.shape)
    sqrt_recip_alpha_t = extract(sqrt_recip_alphas, t, x.shape)

    mean = sqrt_recip_alpha_t * (x - beta_t / sqrt_one_minus_acp_t * eps_pred)

    if t_int == 0:
        return mean
    var = extract(posterior_variance, t, x.shape)
    noise = torch.randn_like(x)
    return mean + var.sqrt() * noise

@torch.no_grad()
def generate(model, n, img_size=IMG_SIZE):
    """Full reverse process from pure noise to image. model is set to eval
    and restored to its previous mode afterward by the caller if needed."""
    model.eval()
    x = torch.randn(n, 3, img_size, img_size, device=DEVICE)
    for t_int in reversed(range(T_STEPS)):
        x = p_sample_step(model, x, t_int)
    return x.clamp(-1, 1)

print('Schedule ready. T =', T_STEPS)

Schedule ready. T = 1000


## 5. U-Net

In [ ]:
def timestep_embedding(t, dim):
    """Sinusoidal embedding"""
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t.float()[:, None] * freqs[None]
    emb = torch.cat([args.sin(), args.cos()], dim=-1)
    if dim % 2 == 1:   # pad if odd
        emb = F.pad(emb, (0, 1))
    return emb

class ResBlock(nn.Module):
    """Two convs with GroupNorm + SiLU, time embedding injected after first conv,
    residual skip with 1x1 conv if channel counts differ."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class Attention(nn.Module):
    """Simple full self-attention over spatial positions, single head for clarity."""
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.q = nn.Conv2d(ch, ch, 1)
        self.k = nn.Conv2d(ch, ch, 1)
        self.v = nn.Conv2d(ch, ch, 1)
        self.proj = nn.Conv2d(ch, ch, 1)
        self.scale = ch ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q = self.q(h).reshape(B, C, H*W).permute(0, 2, 1)   # [B, HW, C]
        k = self.k(h).reshape(B, C, H*W)                     # [B, C, HW]
        v = self.v(h).reshape(B, C, H*W).permute(0, 2, 1)   # [B, HW, C]
        attn = torch.softmax(q @ k * self.scale, dim=-1)     # [B, HW, HW]
        out = (attn @ v).permute(0, 2, 1).reshape(B, C, H, W)
        return x + self.proj(out)

class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.op(x)

class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(ch, ch, 3, padding=1))
    def forward(self, x):
        return self.op(x)

class UNet(nn.Module):
    """
    Standard DDPM U-Net. Predicts noise (epsilon) given a noisy image and timestep.
    Structure: stem conv -> [ResBlock(+Attn) x2 -> Down] per level -> bottleneck
    -> [Up -> ResBlock(+Attn) x2 (with skip concat)] per level -> output conv.
    """
    def __init__(self, base_ch=BASE_CH, ch_mults=CH_MULTS, attn_at=ATTN_AT, img_size=IMG_SIZE):
        super().__init__()
        time_dim = base_ch * 4
        self.time_dim = base_ch
        self.time_mlp = nn.Sequential(
            nn.Linear(base_ch, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim))

        self.stem = nn.Conv2d(3, base_ch, 3, padding=1)

        # ----- encoder -----
        # channels[i] = number of channels entering level i (before that level's blocks)
        self.down_resblocks = nn.ModuleList()
        self.down_attns      = nn.ModuleList()
        self.downsamples     = nn.ModuleList()

        ch = base_ch
        res = img_size
        self.skip_channels = []   # records channel count saved as skip at each block, for decoder

        for level, mult in enumerate(ch_mults):
            out_ch = base_ch * mult
            for _ in range(2):   # 2 resblocks per level
                self.down_resblocks.append(ResBlock(ch, out_ch, time_dim))
                self.down_attns.append(Attention(out_ch) if res in attn_at else nn.Identity())
                ch = out_ch
                self.skip_channels.append(ch)
            if level < len(ch_mults) - 1:
                self.downsamples.append(Down(ch))
                res = res // 2
            else:
                self.downsamples.append(nn.Identity())

        # ----- bottleneck -----
        self.mid_res1 = ResBlock(ch, ch, time_dim)
        self.mid_attn = Attention(ch)
        self.mid_res2 = ResBlock(ch, ch, time_dim)

        # ----- decoder (mirrors encoder, consuming skip_channels in reverse) -----
        self.up_resblocks = nn.ModuleList()
        self.up_attns      = nn.ModuleList()
        self.upsamples     = nn.ModuleList()

        skip_ch_reversed = list(reversed(self.skip_channels))
        for level, mult in reversed(list(enumerate(ch_mults))):
            out_ch = base_ch * mult
            res_here = img_size // (2 ** level) if level < len(ch_mults) - 1 else res
            for _ in range(2):
                skip_ch = skip_ch_reversed.pop(0)
                self.up_resblocks.append(ResBlock(ch + skip_ch, out_ch, time_dim))
                self.up_attns.append(Attention(out_ch) if res_here in attn_at else nn.Identity())
                ch = out_ch
            if level > 0:
                self.upsamples.append(Up(ch))
            else:
                self.upsamples.append(nn.Identity())

        self.out_norm = nn.GroupNorm(8, ch)
        self.out_conv = nn.Conv2d(ch, 3, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(timestep_embedding(t, self.time_dim))
        h = self.stem(x)
        skips = []

        idx = 0
        for level in range(len(self.downsamples)):
            for _ in range(2):
                h = self.down_resblocks[idx](h, t_emb)
                h = self.down_attns[idx](h)
                skips.append(h)
                idx += 1
            h = self.downsamples[level](h)

        h = self.mid_res1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_res2(h, t_emb)

        idx = 0
        for level in range(len(self.upsamples)):
            for _ in range(2):
                skip = skips.pop()
                h = torch.cat([h, skip], dim=1)
                h = self.up_resblocks[idx](h, t_emb)
                h = self.up_attns[idx](h)
                idx += 1
            h = self.upsamples[level](h)

        h = self.out_conv(F.silu(self.out_norm(h)))
        return h

model = UNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params/1e6:.1f}M')

# Quick shape sanity check before training — run this once to catch any
# channel-mismatch bugs immediately instead of after a long training run.
with torch.no_grad():
    test_x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    test_t = torch.randint(0, T_STEPS, (2,), device=DEVICE)
    test_out = model(test_x, test_t)
    assert test_out.shape == test_x.shape, f'Shape mismatch in={test_x.shape} out={test_out.shape}'
    print('Shape check passed:', test_out.shape)

Model parameters: 34.8M
Shape check passed: torch.Size([2, 3, 64, 64])


## 6. EMA

`ema_model` is a fully separate model. After every optimizer step we nudge its
weights toward the live model's weights. **We never copy EMA weights back into
the live `model`.** Sampling during training always uses `ema_model`, and
`model` is left completely alone.

In [ ]:
ema_model = copy.deepcopy(model).to(DEVICE)
for p in ema_model.parameters():
    p.requires_grad_(False)
ema_model.eval()

@torch.no_grad()
def update_ema(ema_model, model, decay=EMA_DECAY):
    for ema_p, p in zip(ema_model.parameters(), model.parameters()):
        ema_p.mul_(decay).add_(p, alpha=1 - decay)
    for ema_b, b in zip(ema_model.buffers(), model.buffers()):
        ema_b.copy_(b)   

print('EMA model created as an independent copy.')

NameError: name 'model' is not defined

## 7. Resume logic

In [ ]:
start_epoch = 0
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

ckpt_path = ""
if ckpt_path is not None:
    ck = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ck['model'])
    ema_model.load_state_dict(ck['ema_model'])
    # Optimizer state matches model state here because we never corrupt model
    # mid-training anymore — safe to resume it for smoother continued training.
    optimizer.load_state_dict(ck['optimizer'])
    start_epoch = ck['epoch'] + 1
    print(f'Resumed from {ckpt_path} — continuing at epoch {start_epoch}')
else:
    print('No checkpoint found — training from scratch.')

Resumed from /kaggle/input/datasets/tahamm786/ddpm-good-10-epochs/outputs/checkpoints/ckpt_epoch0023.pt — continuing at epoch 24


## 8. Training Loop

In [ ]:
def denorm(x):
    return (x.clamp(-1, 1) + 1) / 2

LOG_PATH = f'{OUT_DIR}/log.txt'

def log(msg):
    """Print AND append to a log file on disk, so you can check progress
    mid-run from the Kaggle Output panel without keeping this tab open."""
    print(msg)
    with open(LOG_PATH, 'a') as f:
        f.write(msg + '\n')

def save_sample_grid(epoch):
    # Always sample from ema_model. model is never touched by this function.
    samples = generate(ema_model, n=16)
    grid = make_grid(denorm(samples), nrow=4)
    save_image(grid, f'{OUT_DIR}/samples/epoch_{epoch:04d}.png')
    print(f'  [sample saved: epoch_{epoch:04d}.png]')

def save_checkpoint(epoch, avg_loss):
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'ema_model': ema_model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'loss': avg_loss,
    }, f'{OUT_DIR}/checkpoints/ckpt_epoch{epoch:04d}.pt')
    print(f'  [checkpoint saved: epoch {epoch}]')

scaler = torch.cuda.amp.GradScaler()
loss_history = []

# Sample every epoch for the first 10 epochs (fast early feedback on whether
# training is even pointed the right direction), then fall back to the
# normal SAMPLE_EVERY to avoid slowing down the bulk of training.
EARLY_SAMPLE_EPOCHS = 10

log(f'=== Training started: {NUM_EPOCHS} epochs, starting at epoch {start_epoch} ===')
log(f'Check {OUT_DIR}/latest.png anytime to see the most recent sample.')
log(f'Check {OUT_DIR}/log.txt for the full text log if this view disconnects.')

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()   # ema_model stays in eval() permanently, set once above
    epoch_loss = 0.0
    t_start = time.time()

    for step, x0 in enumerate(train_dl):
        x0 = x0.to(DEVICE)
        B = x0.shape[0]

        t = torch.randint(0, T_STEPS, (B,), device=DEVICE)
        noise = torch.randn_like(x0)
        x_t = q_sample(x0, t, noise)

        with torch.cuda.amp.autocast():
            pred_noise = model(x_t, t)
            loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        update_ema(ema_model, model)   

        epoch_loss += loss.item()
        if step % 300 == 0:
            log(f'  epoch {epoch} step {step}/{len(train_dl)} loss {loss.item():.4f}')

    avg_loss = epoch_loss / len(train_dl)
    loss_history.append(avg_loss)
    log(f'Epoch {epoch} done | avg_loss={avg_loss:.4f} | time={(time.time()-t_start)/60:.1f}min')

    do_sample = (epoch < EARLY_SAMPLE_EPOCHS) or ((epoch + 1) % SAMPLE_EVERY == 0)
    if do_sample:
        save_sample_grid(epoch)

    if (epoch + 1) % SAVE_EVERY == 0:
        save_checkpoint(epoch, avg_loss)

log('Training finished.')

from IPython.display import display
print(f'\nFinal loss this run: {loss_history[-1]:.4f}')
print('Latest sample grid:')
display(Image.open(f'{OUT_DIR}/latest.png'))

=== Training started: 35 epochs, starting at epoch 24 ===
Check /kaggle/working/outputs/latest.png anytime to see the most recent sample.
Check /kaggle/working/outputs/log.txt for the full text log if this view disconnects.


/tmp/ipykernel_23/476487193.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/476487193.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  epoch 24 step 0/6331 loss 0.0101
  epoch 24 step 300/6331 loss 0.0108
  epoch 24 step 600/6331 loss 0.0153
  epoch 24 step 900/6331 loss 0.0114
  epoch 24 step 1200/6331 loss 0.0209
  epoch 24 step 1500/6331 loss 0.0155
  epoch 24 step 1800/6331 loss 0.0110
  epoch 24 step 2100/6331 loss 0.0084
  epoch 24 step 2400/6331 loss 0.0160
  epoch 24 step 2700/6331 loss 0.0132
  epoch 24 step 3000/6331 loss 0.0149
  epoch 24 step 3300/6331 loss 0.0124
  epoch 24 step 3600/6331 loss 0.0117
  epoch 24 step 3900/6331 loss 0.0134
  epoch 24 step 4200/6331 loss 0.0205
  epoch 24 step 4500/6331 loss 0.0149
  epoch 24 step 4800/6331 loss 0.0117
  epoch 24 step 5100/6331 loss 0.0120
  epoch 24 step 5400/6331 loss 0.0102
  epoch 24 step 5700/6331 loss 0.0162
  epoch 24 step 6000/6331 loss 0.0162
  epoch 24 step 6300/6331 loss 0.0222
Epoch 24 done | avg_loss=0.0157 | time=34.6min
  epoch 25 step 0/6331 loss 0.0268
  epoch 25 step 300/6331 loss 0.0136
  epoch 25 step 600/6331 loss 0.0210
  epoch 25 ste

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/outputs/latest.png'

## 9. Loss Curve

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(loss_history)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('Training Loss')
plt.grid(True)
plt.savefig(f'{OUT_DIR}/loss_curve.png', dpi=150)
plt.show()

## 10. Inpainting (RePaint) 

Uses `ema_model`, which was never touched during training. Free-form mask only,
kept simple. No extra training needed.

In [ ]:
def make_freeform_mask(size=IMG_SIZE):
    """1 = keep original pixel, 0 = region to inpaint."""
    mask_img = Image.new('L', (size, size), 255)
    draw = ImageDraw.Draw(mask_img)
    for _ in range(random.randint(4, 8)):
        x0, y0 = random.randint(0, size), random.randint(0, size)
        for _ in range(random.randint(3, 6)):
            x1 = x0 + random.randint(-size//4, size//4)
            y1 = y0 + random.randint(-size//4, size//4)
            draw.line([x0, y0, x1, y1], fill=0, width=random.randint(4, 14))
            x0, y0 = x1, y1
    arr = np.array(mask_img, dtype=np.float32) / 255.0
    return torch.tensor(arr).unsqueeze(0)   # [1, H, W]

@torch.no_grad()
def repaint_inpaint(x0_real, mask, resamples=10):
    ema_model.eval()
    x0_real = x0_real.to(DEVICE)
    mask = mask.to(DEVICE)  # [1,1,H,W] — 1=keep, 0=inpaint
    x = torch.randn_like(x0_real)

    for t_int in reversed(range(T_STEPS)):
        t = torch.full((1,), t_int, device=DEVICE, dtype=torch.long)
        for r in range(resamples):
            # unknown region: one reverse step from model
            x_unknown = p_sample_step(ema_model, x, t_int)
            # known region: sample from forward process at this timestep
            x_known = q_sample(x0_real, t, torch.randn_like(x0_real))
            # combine: keep known region clean, fill unknown with model output
            x = mask * x_known + (1 - mask) * x_unknown
            # jump forward again for next resample (except on last resample)
            if r < resamples - 1 and t_int < T_STEPS - 1:
                x = (1 - betas[t_int+1]).sqrt() * x + betas[t_int+1].sqrt() * torch.randn_like(x)
    return x.clamp(-1, 1)

# demo 
demo_batch = next(iter(train_dl))[:4].to(DEVICE)
mask = make_freeform_mask().unsqueeze(0).to(DEVICE)    # [1,1,H,W]

results = []
for i in range(demo_batch.shape[0]):
    out = repaint_inpaint(demo_batch[i:i+1], mask)
    results.append(out)

rows = []
for orig, res in zip(demo_batch, results):
    masked = orig.unsqueeze(0) * mask[0]
    rows += [orig.unsqueeze(0), masked, res]

def denorm(x):
    return (x.clamp(-1, 1) + 1) / 2
    
grid = make_grid(denorm(torch.cat(rows)), nrow=3)
save_image(grid, f'{OUT_DIR}/inpaint_demo.png')

plt.figure(figsize=(8, 10))
plt.imshow(denorm(grid).cpu().permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('original | masked | inpainted')
plt.show()

In [ ]:
#Generating images

n_images = 4   

with torch.no_grad():
    samples = generate(ema_model, n=n_images)

grid = make_grid(denorm(samples), nrow=4)
plt.figure(figsize=(10, 10))
plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
plt.axis('off')
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display

demo_batch = next(iter(train_dl))[:1].to(DEVICE)   #  one face at a time

plt.figure(figsize=(4,4))
plt.imshow(denorm(demo_batch[0]).cpu().permute(1,2,0).numpy())
plt.title('Face to inpaint — note pixel coords below')
plt.show()

x_slider = widgets.IntRangeSlider(value=[20,44], min=0, max=64, description='X range')
y_slider = widgets.IntRangeSlider(value=[20,44], min=0, max=64, description='Y range')
ipy_display(x_slider, y_slider)

In [ ]:
x0, x1 = x_slider.value
y0, y1 = y_slider.value

mask = torch.ones(1, 1, IMG_SIZE, IMG_SIZE)
mask[:, :, y0:y1, x0:x1] = 0   # 0 = erase this region
mask = mask.to(DEVICE)

result = repaint_inpaint(demo_batch, mask, resamples=10)

masked_preview = demo_batch * mask
grid = make_grid(denorm(torch.cat([demo_batch, masked_preview, result])), nrow=3)

plt.figure(figsize=(9,3))
plt.imshow(grid.cpu().permute(1,2,0).numpy())
plt.axis('off')
plt.title('original | masked | inpainted')
plt.show()